# Round 13 — Localized passage evidence against labeled supports

Two new feature families; 48 primary columns. Fixed classifier and cached anchor. This is exploratory development, not a Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/passage_support_features.json").is_file())
from scripts.run_passage_support_features import figures, write_dashboard
SPEC = json.loads((ROOT / "configs/passage_support_features.json").read_text())
RESULT = json.loads((ROOT / "reports/passage_support_features/results.json").read_text())
print("Registered primary:", SPEC["primary"])
print("New classifier fits:", RESULT["new_fits"])
print("Prior readouts reused:", RESULT["reused_prior_controls"])
CHARTS = figures(RESULT)

Registered primary: passage_all
New classifier fits: 12
Prior readouts reused: 10


## 1. Hypothesis and unchanged anchor

Does a localized phrase provide evidence that is diluted when a whole comment is represented by one vector?

24 word-passage and 24 character-passage summaries, including first/last evidence and strongest-evidence location. The context anchor was selected after Round 9 and was not promoted.

In [2]:
display(pd.DataFrame(RESULT["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## 2. Conditional uncertainty

Per-policy results matter. Intervals cover registered within-round contrasts, not all project-wide adaptive decisions.

In [3]:
display(pd.DataFrame(RESULT["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,word_passages,context_evidence,word_passages vs context anchor,0.000998,-0.023293,0.025289
1,character_passages,context_evidence,character_passages vs context anchor,-0.004576,-0.028867,0.019715
2,passage_all,context_evidence,passage_all vs context anchor,-0.004873,-0.029164,0.019418
3,whole_comment_all,context_evidence,whole_comment_all vs context anchor,-0.002308,-0.026599,0.021983
4,boundary_null_all,context_evidence,boundary_null_all vs context anchor,-0.005822,-0.030113,0.018469
5,label_null_all,context_evidence,label_null_all vs context anchor,-0.014111,-0.038402,0.010180
6,passage_all,qwen_raw,Primary vs qwen_raw,0.004491,-0.019800,0.028782
7,passage_all,frozen_basic,Primary vs frozen_basic,0.001316,-0.022975,0.025607
8,passage_all,uniform_all,Primary vs uniform_all,-0.004526,-0.028817,0.019765
9,passage_all,whole_comment_all,Primary vs whole_comment_all,-0.002565,-0.026856,0.021726


## 3. Mechanism controls and family ablations

whole_comment_all, boundary_null_all, label_null_all. No secondary winner silently replaces the primary.

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## 4. Coverage and reference diagnostics

Whole-reference labels remain document labels. No sentence label is invented; no tail passage is silently discarded.

In [5]:
display(pd.DataFrame(RESULT["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,mode,family,reference_rows,vocabulary_columns,mean_passages,multiple_passage_fraction,zero_lexical_fraction,maximum_passages,new_passage_labels,query_used_for_vocabulary
0,0,0,0,passages,word,419,6000,2.338095,0.585714,0.042770,12,0,False
1,0,0,0,whole_comment,word,419,6000,1.000000,0.585714,0.000000,1,0,False
2,0,0,0,boundary_null,word,419,6000,2.338095,0.585714,0.016293,12,0,False
3,0,0,0,label_null,word,419,6000,2.338095,0.585714,0.042770,12,0,False
4,0,0,0,passages,character,419,6000,2.338095,0.585714,0.004073,12,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,outer_query,0,label_null,word,366,6000,3.006182,0.825348,0.026221,11,0,False
60,1,outer_query,0,passages,character,366,6000,3.006182,0.825348,0.006684,11,0,False
61,1,outer_query,0,whole_comment,character,366,6000,1.000000,0.825348,0.000000,1,0,False
62,1,outer_query,0,boundary_null,character,366,6000,3.006182,0.825348,0.000000,11,0,False


## 5. Fitted associations

Coefficients are not causal effects. Use matched additions/removals to judge evidence.

In [6]:
CHARTS[6].show(renderer="plotly_mimetype")

## 6. Probability quality and metrics

AUC, per-policy macro AUC, and ranked pooled AUC are distinct. Brier/log loss track probability quality. None is a new hidden Kaggle score.

In [7]:
display(pd.DataFrame(RESULT["pooled_metrics"]))
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,word_passages,0.730255,0.738013,0.744573
6,character_passages,0.724681,0.734648,0.738145
7,passage_all,0.724384,0.735518,0.739711
8,whole_comment_all,0.726949,0.739874,0.742511
9,boundary_null_all,0.723435,0.733203,0.737058


## 7. Fixed decision and limitations

Primary: `passage_all`. Both companion designs are independent. The inherited training Qwen answer margin remains in-sample even though these new features are cross-fitted.

Research: https://aclanthology.org/C16-1220/

In [8]:
print("Decision:", RESULT["decision"])
for item in RESULT["primary_requirements"]:
    print(item["reference"], item["per_policy_delta"], "passed:", item["passed"])
for limitation in RESULT["limitations"]:
    print(limitation)
print("Interactive dashboard:", write_dashboard(ROOT, RESULT))

Decision: DO_NOT_PROMOTE_PRIMARY
qwen_raw [0.012425373134328233, -0.0034441596054870516] passed: False
frozen_basic [-0.0014179104477614057, 0.0040508013541809] passed: False
context_evidence [-0.011791044776119475, 0.0020449697657580757] passed: False
uniform_all [-0.011567164179104528, 0.002514627893779098] passed: False
whole_comment_all [-0.001940298507462801, -0.0031897614528091367] passed: False
boundary_null_all [-0.0026119402985076423, 0.004510674937868164] passed: False
label_null_all [0.008955223880596996, 0.009520361636758623] passed: False
Exploratory adaptive development on the same 881 queries, not a fresh holdout or Kaggle score.
The context_evidence anchor is held fixed, selected after Round 9, and was never promoted.
New reference features are text-group cross-fitted. The inherited adapted answer score is in-sample on support labels.
Cross-fitting the new features cannot repair that inherited answer-score limitation.
Simultaneous intervals cover this round, not the ful

Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/passage_support_features/dashboard.html
